# NB01 — NumPy Vectorization and Broadcasting

**Module 17: HPC, Parallel Computing, and Rust**  
**Tier 2 — Working Competence**

---

## 1. Motivation

You have 10,000 DNA sequences. You want to compute the GC content of each one. The naive approach is a Python `for` loop over characters — it works, but for 10,000 sequences of length 1,000 that is 10 million character comparisons executed one at a time in the Python interpreter.

NumPy vectorization moves that loop into compiled C code. The same 10 million operations take milliseconds instead of seconds — without writing a single line of C, Cython, or Rust.

This notebook teaches you to recognize the pattern "this loop can be vectorized" and to carry out the translation reliably.

---

## 2. Intuition

Think of a NumPy array as a **vector in the mathematical sense**: a single object that represents many numbers at once. When you write `a + b` for two NumPy arrays, NumPy adds every pair of elements simultaneously — not in a Python loop, but in a single compiled C call.

**Broadcasting** extends this to arrays of different shapes. Instead of requiring both arrays to be the same shape, NumPy "stretches" smaller arrays along missing dimensions so that the operation is still element-wise. The stretching is conceptual — no memory is copied.

---

## 3. Biological background

**GC content** is the fraction of guanine (G) and cytosine (C) bases in a DNA sequence. It is a basic but biologically meaningful statistic:
- High GC content → stronger base stacking, higher melting temperature
- Correlates with gene density in mammalian genomes (GC-rich isochores have more genes)
- Used as a quality-control signal in NGS: extreme GC biases indicate PCR artefacts

**Pairwise Hamming distance** counts the number of positions where two equal-length sequences differ. For aligned sequences, it is a proxy for evolutionary distance.

Both operations are computed over large arrays constantly in bioinformatics pipelines. Vectorizing them is not premature optimization — it is the baseline expectation.

---

## 4. Mathematical explanation

### GC content (vectorized)

Let $S$ be a matrix of shape $(N, L)$ where each row is a sequence encoded as integers: A=0, T=1, C=2, G=3.

$$\text{gc}_i = \frac{1}{L} \sum_{j=0}^{L-1} \mathbf{1}[S_{ij} \in \{2, 3\}]$$

In NumPy: `((S == 2) | (S == 3)).mean(axis=1)` — one expression, no loop.

### Pairwise Hamming distance

For sequences $a$ and $b$ of length $L$:

$$d_H(a, b) = \sum_{j=0}^{L-1} \mathbf{1}[a_j \neq b_j]$$

For the full $N \times N$ matrix, naive complexity is $O(N^2 L)$.

With broadcasting: expand $S$ to shape $(N, 1, L)$ and $(1, N, L)$, compare, sum — still $O(N^2 L)$ but in compiled code.

### Broadcasting rules

NumPy aligns shapes from the **right**. Dimensions are compatible if they are equal or one of them is 1. A dimension of size 1 is "broadcast" (virtually repeated) to match the other.

| Shape A | Shape B | Result |
|---------|---------|--------|
| (N, L)  | (L,)    | (N, L) — B broadcast along axis 0 |
| (N, 1, L) | (1, N, L) | (N, N, L) — both broadcast |
| (3,)    | (4,)    | **Error** — incompatible sizes |

---

## 5. Computational explanation

The key pattern:

```
Python loop        → NumPy array operation
for i in range(N): → axis argument (axis=0 or axis=1)
  result[i] = f(x[i])  → result = f(x)  (vectorized f)
```

For pairwise operations, the key trick is **adding a new axis** to create broadcastable shapes:

```python
S[:, np.newaxis, :]  # shape (N, 1, L)
S[np.newaxis, :, :]  # shape (1, N, L)
# Comparison broadcasts to (N, N, L)
```

`np.einsum` provides Einstein summation notation — a compact way to express contractions (dot products, outer products, traces) without intermediate arrays.

---

## 6. Python implementation

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt

np.random.seed(42)
print("NumPy version:", np.__version__)

In [ ]:
# --- Data generation ---
# Encode sequences as integers: A=0, T=1, C=2, G=3
def generate_sequences(N, L, seed=42):
    """Generate N random DNA sequences of length L as integer-encoded array."""
    rng = np.random.default_rng(seed)
    return rng.integers(0, 4, size=(N, L), dtype=np.int8)

S_small = generate_sequences(N=100, L=200)
S_large = generate_sequences(N=10_000, L=200)
print(f"Small array shape: {S_small.shape}, dtype: {S_small.dtype}")
print(f"Large array shape: {S_large.shape}, Memory: {S_large.nbytes / 1e6:.1f} MB")

In [ ]:
# --- GC content: Python loop vs NumPy ---

def gc_content_python(seqs):
    """Naive Python loop: one sequence at a time."""
    result = []
    for seq in seqs:
        gc = sum(1 for b in seq if b == 2 or b == 3)
        result.append(gc / len(seq))
    return np.array(result)

def gc_content_numpy(seqs):
    """Vectorized: boolean mask then mean over axis=1."""
    return ((seqs == 2) | (seqs == 3)).mean(axis=1)

# Correctness check on small array
gc_py = gc_content_python(S_small)
gc_np = gc_content_numpy(S_small)
assert np.allclose(gc_py, gc_np), "Implementations disagree!"
print(f"GC content range (N=100): [{gc_np.min():.3f}, {gc_np.max():.3f}]")
print("Correctness check: PASSED")

In [ ]:
# --- Timing: GC content at different N ---
Ns = [100, 500, 1000, 5000, 10000]
times_python = []
times_numpy = []

for N in Ns:
    S = generate_sequences(N, L=200)
    
    # Python loop
    if N <= 2000:  # skip large N for slow loop
        t0 = time.perf_counter()
        for _ in range(3):
            gc_content_python(S)
        times_python.append((time.perf_counter() - t0) / 3)
    else:
        # Estimate from smaller N (linear scaling)
        times_python.append(times_python[-1] * (N / Ns[Ns.index(N)-1]))
    
    # NumPy
    t0 = time.perf_counter()
    for _ in range(10):
        gc_content_numpy(S)
    times_numpy.append((time.perf_counter() - t0) / 10)

print(f"{'N':>8} {'Python (s)':>12} {'NumPy (s)':>12} {'Speedup':>10}")
for N, tp, tn in zip(Ns, times_python, times_numpy):
    print(f"{N:>8} {tp:>12.4f} {tn:>12.4f} {tp/tn:>10.1f}x")

In [ ]:
# --- Pairwise Hamming distance: Python loop vs NumPy broadcasting ---

def hamming_python(seqs):
    """O(N^2 L) pure Python — baseline."""
    N = len(seqs)
    result = np.zeros((N, N), dtype=np.float32)
    for i in range(N):
        for j in range(i + 1, N):
            d = int(np.sum(seqs[i] != seqs[j]))
            result[i, j] = result[j, i] = d
    return result

def hamming_numpy(seqs):
    """
    NumPy broadcasting:
    seqs[:, np.newaxis, :]  → shape (N, 1, L)
    seqs[np.newaxis, :, :]  → shape (1, N, L)
    comparison              → shape (N, N, L) boolean
    sum over axis=2         → shape (N, N)
    """
    diff = seqs[:, np.newaxis, :] != seqs[np.newaxis, :, :]  # (N, N, L) bool
    return diff.sum(axis=2).astype(np.float32)

# Correctness check
S10 = generate_sequences(N=10, L=50)
H_py = hamming_python(S10)
H_np = hamming_numpy(S10)
assert np.allclose(H_py, H_np), "Hamming implementations disagree!"
print("Hamming correctness: PASSED")
print(f"Sample distances (first 5x5):\n{H_np[:5,:5].astype(int)}")

In [ ]:
# --- Sliding window mean: Python vs NumPy ---

def sliding_mean_python(signal, w):
    """Sliding window mean — Python loop."""
    n = len(signal)
    result = np.empty(n - w + 1)
    for i in range(n - w + 1):
        result[i] = signal[i:i+w].mean()
    return result

def sliding_mean_numpy(signal, w):
    """
    Cumulative sum trick:
      cumsum[i+w] - cumsum[i]  gives the sum of window [i, i+w)
      Divide by w for the mean.
    O(N) time and O(N) space — optimal.
    """
    cs = np.cumsum(signal)
    cs = np.concatenate(([0], cs))
    return (cs[w:] - cs[:-w]) / w

signal = np.random.randn(100_000)
w = 100

# Correctness
sm_py = sliding_mean_python(signal[:1000], w)
sm_np = sliding_mean_numpy(signal[:1000], w)
assert np.allclose(sm_py, sm_np), "Sliding mean disagrees!"
print("Sliding mean correctness: PASSED")

# Timing
t0 = time.perf_counter()
sliding_mean_python(signal[:10_000], w)
t_py = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(10):
    sliding_mean_numpy(signal, w)
t_np = (time.perf_counter() - t0) / 10

print(f"Python loop (N=10k): {t_py:.4f} s")
print(f"NumPy cumsum (N=100k): {t_np:.4f} s  (larger N, still faster)")

In [ ]:
# --- np.einsum for pairwise dot products ---
# Context: comparing gene expression profiles as cosine similarity numerators

X = np.random.randn(200, 50).astype(np.float32)  # 200 samples, 50 genes

# Method 1: explicit loop
dots_loop = np.zeros((200, 200), dtype=np.float32)
for i in range(200):
    for j in range(200):
        dots_loop[i, j] = np.dot(X[i], X[j])

# Method 2: @ operator (matrix multiply)
dots_matmul = X @ X.T  # identical to all pairwise dot products

# Method 3: np.einsum — reads as "for each i,j: sum over k of X[i,k]*X[j,k]"
dots_einsum = np.einsum('ik,jk->ij', X, X)

assert np.allclose(dots_loop, dots_matmul, atol=1e-4)
assert np.allclose(dots_loop, dots_einsum, atol=1e-4)
print("einsum correctness: PASSED")
print("np.einsum('ik,jk->ij', X, X) reads:")
print("  For each output position (i,j):")
print("  Multiply X[i,k] * X[j,k] and sum over k")
print("  This is exactly X @ X.T, expressed as index notation")

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Execution time vs N (log-log)
ax = axes[0]
ax.loglog(Ns, times_python, 'o-', color='#e74c3c', label='Python loop', linewidth=2)
ax.loglog(Ns, times_numpy, 's-', color='#2980b9', label='NumPy vectorized', linewidth=2)
ax.set_xlabel('Number of sequences (N)', fontsize=11)
ax.set_ylabel('Wall time (seconds)', fontsize=11)
ax.set_title('GC Content: Execution Time vs N', fontweight='bold')
ax.legend()
ax.grid(True, which='both', alpha=0.3)

# Panel 2: Speedup ratio
ax = axes[1]
speedups = [tp/tn for tp, tn in zip(times_python, times_numpy)]
ax.bar(range(len(Ns)), speedups, color='#27ae60', edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(Ns)))
ax.set_xticklabels([str(n) for n in Ns], fontsize=9)
ax.set_xlabel('Number of sequences (N)', fontsize=11)
ax.set_ylabel('Speedup (Python time / NumPy time)', fontsize=11)
ax.set_title('NumPy Speedup Over Python Loop', fontweight='bold')
for i, s in enumerate(speedups):
    ax.text(i, s + 1, f'{s:.0f}x', ha='center', va='bottom', fontweight='bold', fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

# Panel 3: Broadcasting shape diagram
ax = axes[2]
ax.axis('off')
diagram_text = (
    "Broadcasting Rules\n"
    "(shapes aligned from RIGHT)\n\n"
    "Example: GC content\n"
    "  S     shape: (N, L)\n"
    "  axis=1 → reduce over L\n"
    "  result shape: (N,)\n\n"
    "Example: Pairwise Hamming\n"
    "  S[:, np.newaxis, :]  → (N, 1, L)\n"
    "  S[np.newaxis, :, :]  → (1, N, L)\n"
    "  broadcast → (N, N, L)\n"
    "  .sum(axis=2) → (N, N)\n\n"
    "Rule: dims compatible if\n"
    "  equal OR one of them = 1"
)
ax.text(0.05, 0.95, diagram_text, transform=ax.transAxes,
        fontsize=10, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.set_title('Broadcasting Shape Diagram', fontweight='bold')

plt.tight_layout()
plt.savefig('../datasets/nb01_vectorization.png', dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved.")

## 8. Exercises

Attempt these before looking at any solutions. Write your answers in `exercises/ex01_numpy_vectorization.py`.

1. **Sliding GC content:** Implement a vectorized sliding window GC content function using the cumsum trick. Input: a 1D integer-encoded sequence array. Output: GC fraction for each window position.

2. **Normalized pairwise dot product (cosine similarity):** Given the pairwise dot product matrix from `np.einsum`, compute the full cosine similarity matrix in one vectorized expression (no loop). Hint: you need the norm of each row.

3. **AT-richness filter:** Given a matrix `S` of shape (N, L), return the indices of sequences where AT content > 0.6, using only NumPy boolean indexing and `.mean(axis=1)`. No loops.

4. **Broadcasting error diagnosis:** For each pair of shapes below, predict whether NumPy will broadcast them successfully and what the output shape will be. Then verify in code.
   - (100, 1) and (1, 200)
   - (100, 200) and (200,)
   - (100, 200) and (100,)
   - (3, 1, 200) and (1, 50, 200)

## 9–11. Implementation → Review → Improve

*(Complete in the exercises file, then bring back here for review.)*

## 12. Reflection

*Write a paragraph here after completing the exercises: What is broadcasting, in your own words? When does it fail? What was the most surprising speedup you saw?*

---

## Papers referenced

- Harris, C.R. et al. (2020). "Array programming with NumPy." *Nature* 585:357–362. — foundational design paper for NumPy's vectorization model.

## References

- NumPy broadcasting docs: https://numpy.org/doc/stable/user/basics.broadcasting.html
- `np.einsum` tutorial: https://numpy.org/doc/stable/reference/generated/numpy.einsum.html

## Future work / open questions

- How does NumPy's broadcasting compare to Numba's auto-parallelization (NB04)?
- For the Hamming distance matrix at N=10,000, the (N, N, L) intermediate array would be 10,000 × 10,000 × 200 bytes = 20 GB — broadcasting has a memory cost. What alternative exists? (Hint: compute in blocks, or use Numba with prange.)